# Détection de maladies du pommier — EfficientNet-B4

**Avant de commencer :** `Runtime → Change runtime type → T4 GPU`

**Dataset :** [PlantVillage sur Kaggle](https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset)

**Classes réelles (pommier) :**
| Classe | Nombre d'images |
|--------|----------------|
| Apple___Apple_scab | ~2 016 |
| Apple___Black_rot | ~1 180 |
| Apple___Cedar_apple_rust | ~275 ← **rare** |
| Apple___healthy | ~1 645 |

> **Note :** Le déséquilibre Cedar Rust est géré par pondération des classes.

In [ ]:
# ── 1. Installer les dépendances ──────────────────────────────────────────────
!pip install -q kaggle scikit-learn

In [ ]:
# ── 2. Télécharger le dataset (uploader kaggle.json depuis Kaggle > Account) ──
from google.colab import files
print('Upload ton fichier kaggle.json')
files.upload()

import os, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('kaggle.json', '/root/.kaggle/')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!kaggle datasets download -d abdallahalidev/plantvillage-dataset -p /content/
!unzip -q /content/plantvillage-dataset.zip -d /content/plantvillage/

In [ ]:
# ── 3. Extraire uniquement les 4 classes pommier ──────────────────────────────
import shutil
from pathlib import Path

APPLE_CLASSES = [
    'Apple___Apple_scab',
    'Apple___Black_rot',
    'Apple___Cedar_apple_rust',
    'Apple___healthy',
]

src_root = Path('/content/plantvillage/plantvillage dataset/color')
dst_root = Path('/content/apple_dataset')

for cls in APPLE_CLASSES:
    src = src_root / cls
    dst = dst_root / cls
    if src.exists():
        shutil.copytree(src, dst)
        print(f'{cls}: {len(list(dst.iterdir()))} images')
    else:
        print(f'MANQUANT: {src}')

In [ ]:
# ── 4. Configuration ──────────────────────────────────────────────────────────
IMAGE_SIZE   = 380   # EfficientNet-B4 natif (PAS 224 — c'est B0)
BATCH_SIZE   = 16
EPOCHS_HEAD  = 15
EPOCHS_FINE  = 35
LR_HEAD      = 1e-3
LR_FINE      = 5e-5  # ×20 plus faible pour éviter catastrophic forgetting
DROPOUT      = 0.4
DATA_DIR     = '/content/apple_dataset'

APPLE_CLASSES = [
    'Apple___Apple_scab',
    'Apple___Black_rot',
    'Apple___Cedar_apple_rust',
    'Apple___healthy',
]
NUM_CLASSES = len(APPLE_CLASSES)

In [ ]:
# ── 5. Dataset + Augmentation ─────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

common_args = dict(
    directory=DATA_DIR,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_names=APPLE_CLASSES,
    label_mode='categorical',
)
train_ds = keras.utils.image_dataset_from_directory(
    validation_split=0.2, subset='training', seed=42, **common_args)
val_ds   = keras.utils.image_dataset_from_directory(
    validation_split=0.2, subset='validation', seed=42, **common_args)

augment = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.15),
    layers.RandomBrightness(0.2),
    layers.RandomContrast(0.2),
])

AUTOTUNE = tf.data.AUTOTUNE
train_ds = (train_ds
            .map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
            .prefetch(AUTOTUNE))
val_ds   = val_ds.prefetch(AUTOTUNE)

In [ ]:
# ── 6. Pondération des classes (Cedar Rust très rare) ─────────────────────────
from sklearn.utils.class_weight import compute_class_weight

all_labels = []
for _, y in train_ds:
    all_labels.extend(np.argmax(y.numpy(), axis=1))

weights = compute_class_weight('balanced',
                               classes=np.arange(NUM_CLASSES),
                               y=np.array(all_labels))
class_weights = {i: w for i, w in enumerate(weights)}
print('Poids des classes:', class_weights)
# Cedar Apple Rust va avoir un poids ~4-6× plus élevé

In [ ]:
# ── 7. Phase 1 : Tête seule (base gelée) ─────────────────────────────────────
from tensorflow.keras.applications import EfficientNetB4

def build_model(trainable_base=False, fine_tune_from_pct=0.0):
    base = EfficientNetB4(
        include_top=False,
        weights='imagenet',
        input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
        pooling='avg'
    )
    base.trainable = trainable_base
    if trainable_base and fine_tune_from_pct > 0:
        freeze_until = int(len(base.layers) * fine_tune_from_pct)
        for layer in base.layers[:freeze_until]:
            layer.trainable = False
        print(f'Couches entraînables: {sum(1 for l in base.layers if l.trainable)}/{len(base.layers)}')
    
    inputs  = keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
    x       = base(inputs, training=trainable_base)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dropout(DROPOUT)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    return keras.Model(inputs, outputs)

callbacks_phase1 = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7),
    keras.callbacks.ModelCheckpoint('best_phase1.h5', monitor='val_accuracy', save_best_only=True),
]

model = build_model(trainable_base=False)
model.compile(
    optimizer=keras.optimizers.Adam(LR_HEAD),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print('Phase 1: entraînement tête seule…')
history1 = model.fit(train_ds, validation_data=val_ds,
                     epochs=EPOCHS_HEAD, class_weight=class_weights,
                     callbacks=callbacks_phase1)

In [ ]:
# ── 8. Phase 2 : Fine-tuning (30% dernières couches) ─────────────────────────
callbacks_phase2 = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=4, min_lr=1e-8),
    keras.callbacks.ModelCheckpoint('best_phase2.h5', monitor='val_accuracy', save_best_only=True),
]

model = build_model(trainable_base=True, fine_tune_from_pct=0.70)
model.load_weights('best_phase1.h5')
model.compile(
    optimizer=keras.optimizers.Adam(LR_FINE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print('Phase 2: fine-tuning partiel…')
history2 = model.fit(train_ds, validation_data=val_ds,
                     epochs=EPOCHS_FINE, class_weight=class_weights,
                     callbacks=callbacks_phase2)

model.save('apple_disease_final.h5')
print('Modèle sauvegardé : apple_disease_final.h5')

In [ ]:
# ── 9. Courbes d'entraînement ─────────────────────────────────────────────────
import matplotlib.pyplot as plt

acc  = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss = history1.history['val_loss']     + history2.history['val_loss']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(acc);  ax1.set_title('Val Accuracy');  ax1.set_ylim(0, 1)
ax2.plot(loss); ax2.set_title('Val Loss')
ax1.axvline(EPOCHS_HEAD, color='red', linestyle='--', label='début fine-tuning')
ax2.axvline(EPOCHS_HEAD, color='red', linestyle='--')
ax1.legend()
plt.tight_layout()
plt.savefig('training_curves.png', dpi=120)
plt.show()
print(f'Meilleure val_accuracy: {max(acc):.4f}')

In [ ]:
# ── 10. Conversion TFLite Float-16 ───────────────────────────────────────────
import json

model = keras.models.load_model('apple_disease_final.h5')

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]   # Float-16, pas INT8

tflite_model = converter.convert()

with open('apple_disease_f16.tflite', 'wb') as f:
    f.write(tflite_model)

with open('class_names.json', 'w', encoding='utf-8') as f:
    json.dump(APPLE_CLASSES, f, ensure_ascii=False, indent=2)

size_mb = len(tflite_model) / 1_000_000
print(f'Taille Float-16: {size_mb:.1f} MB')
print('Fichiers générés: apple_disease_f16.tflite + class_names.json')

In [ ]:
# ── 11. Télécharger les fichiers ──────────────────────────────────────────────
from google.colab import files
files.download('apple_disease_f16.tflite')
files.download('class_names.json')
files.download('training_curves.png')
print('Copier ces fichiers dans : flutter_app/assets/model/')